# BioVision AI — Train real disease-classification models in Google Colab

Run this notebook on your phone or any browser; a laptop is not required. It downloads real Kaggle datasets, trains the repository's MobileNetV2 models, and packages the resulting model weights and class maps.

**Security:** The Kaggle token previously pasted into ChatGPT should be revoked immediately. Create a fresh token and enter it only in the password prompt below. Never paste tokens into chat, notebook source, GitHub, or screenshots. Human skin-lesion data is non-commercial and this model is not a medical diagnostic tool.

**Compute/time:** Training all three domains can take a long time and may exceed a free Colab session. If it stops, change `DOMAINS` to just `['plant']` and run one domain at a time. Review each dataset's license and labels before using it.

In [ ]:
# Install dependencies (Colab's Python version may differ from local requirements)
%pip -q install kaggle streamlit opencv-python-headless pillow pandas scikit-learn matplotlib plotly seaborn reportlab pyttsx3 tqdm scipy
%pip -q install --upgrade tensorflow


In [ ]:
# Enter a NEW Kaggle API token privately. Input is hidden and not written to this notebook.
import os, getpass
token = getpass.getpass('Paste your NEW Kaggle API token (input hidden): ').strip()
if not token.startswith('KGAT_'):
    raise ValueError('This notebook expects a Kaggle API token beginning with KGAT_. Generate a new token in Kaggle Settings > API.')
os.environ['KAGGLE_API_TOKEN'] = token
del token
print('Token set for this runtime; it was not saved into the notebook.')


In [ ]:
# Get the project source
%cd /content
!rm -rf AI-Multi-Domain-Disease-Prediction
!git clone --depth 1 https://github.com/247r1a05c2-b/AI-Multi-Domain-Disease-Prediction.git
%cd /content/AI-Multi-Domain-Disease-Prediction/BioVisionAI
print('Project ready:', os.getcwd())


## Choose domains and train

Default below attempts all three domains. If the session runs out of time or storage, run one domain at a time. Dataset availability, Kaggle access, licensing, and Colab resource limits can affect completion.

In [ ]:
import subprocess, sys
DOMAINS = ['plant', 'human', 'animal']  # or ['plant'] to start with the smaller first task
for domain in DOMAINS:
    cmd = [sys.executable, 'training/prepare_real_datasets.py', '--domains', domain, '--train']
    print('\nTraining domain:', domain, '\nCommand:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
    print('Finished domain:', domain)


In [ ]:
# Verify real trained model artifacts, then package the complete app source + trained weights
from pathlib import Path
import zipfile
model_dir = Path('saved_models')
artifacts = sorted(p for p in model_dir.glob('*') if p.is_file() and (p.suffix in {'.h5', '.keras', '.json'}))
for p in artifacts:
    print(p.name, f'{p.stat().st_size / (1024*1024):.1f} MiB')
models = [p for p in artifacts if p.suffix in {'.h5', '.keras'}]
if not models:
    raise RuntimeError('No trained model file found; do not treat this run as successful.')
project_root = Path.cwd()
zip_path = '/content/BioVisionAI_full_project_with_models.zip'
# Exclude large datasets, downloaded archives, caches, logs, local DB and credentials.
excluded_dirs = {'datasets', 'downloads', '__pycache__', '.git', '.streamlit_cache'}
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in project_root.rglob('*'):
        if not p.is_file():
            continue
        rel = p.relative_to(project_root)
        if any(part in excluded_dirs for part in rel.parts):
            continue
        if rel.parts and rel.parts[0] in {'logs', 'reports', 'database'}:
            continue
        if p.name in {'kaggle.json'} or 'token' in p.name.lower() or 'secret' in p.name.lower():
            continue
        z.write(p, arcname=str(Path('BioVisionAI') / rel))
print('Complete project archive created:', zip_path)
print('Contains app source, dependencies, training scripts, and verified model files/classes. Datasets and credentials are excluded.')
from google.colab import files
files.download(zip_path)


## Important deployment note

The final ZIP is built only after at least one actual `.h5` or `.keras` trained model exists. It contains the BioVisionAI application source, dependencies, training scripts, and any generated model weights/class maps; it excludes downloaded datasets, local database/logs, and credentials. Training may still fail due to Kaggle dataset permissions, labels, or free-runtime limits. No model is automatically uploaded to GitHub or deployed to Streamlit. Human skin-lesion predictions are educational only and must not be used for diagnosis. The HAM10000 dataset has non-commercial restrictions.